OpenAPI 문제

In [62]:
from dotenv import load_dotenv
import os

load_dotenv()

# 사용할 키 가져오기
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')


In [63]:
from openai import OpenAI

In [64]:
client = OpenAI(api_key=OPENAI_API_KEY)

In [65]:
import requests
from datetime import datetime, date

def calculate_age(birthdate):
    birth = datetime.strptime(birthdate, '%Y-%m-%d').date()
    today = date.today()

    age = today.year - birth.year

    if(today.month, today.day) < (birth.month, today.day):
        age -= 1

    return age

def convert_currency(amount):
    rate = 1330
    if amount < 0 :
        return print('수량음 음수일 수 없습니다.')
    
    krw = rate*amount

    return krw

def calculate_bmi(height, weight):
    height_m = height/100
    bmi = weight/(height_m**2)
    
    return bmi

In [66]:
fn_tools = [
    {
        'type' : 'function',
        'name' : 'calculate_age',
        'description' : '나이를 YYYY-MM-DD 형태로 받으면 만 나이를 계산하는 함수',
        'parameters' : {
            'type' : 'object',
            'properties' : {
                'birthdate' : {'type':'string', 'description' : 'YYYY-MM-DD'}
            },
        'required' : ['birthdate'],
        'additionalProperties' : False
        },
        'strict' : True
    },
    {
        'type' : 'function',
        'name' : 'convert_currency',
        'description' : '달러 수량을 입력하면 환율을 적용해 원화로 반환한다.',
        'parameters' : {
            'type' : 'object',
            'properties' : {
                'amount' : {'type':'number'}
            },
        'required' : ['amount'],
        'additionalProperties' : False
        },
        'strict' : True
    },
    {
        'type' : 'function',
        'name' : 'calculate_bmi',
        'description' : '키와 몸무게를 이용해 BMI 지수를 계산한다.',
        'parameters' : {
            'type' : 'object',
            'properties' : {
                'height' : {'type':'number', 'description':'키(cm)'},
                'weight' : {'type':'number', 'description':'몸무게(kg)'}
            },
        'required' : ['height', 'weight'],
        'additionalProperties' : False
        },
        'strict' : True
    }
]


In [67]:
input_messages = [
    {
        'role' : 'user',
        'content' : '내 생일은 1999-10-12일이야. 그리고 내가 100달러를 가지고 있는데 원화로 얼마인지 알려줘. 그리고 내 키가 190에 몸무게가 100kg인데 BMI 지수를 알려줘.'
    }
]

response = client.responses.create(
    model = 'gpt-5.5',
    input = input_messages,
    tools = fn_tools
)

In [68]:
result = calculate_age('1992-05-01')
print(result)

34


In [69]:
response.output

[ResponseFunctionToolCall(arguments='{"birthdate":"1999-10-12"}', call_id='call_vkm4kNFDdBGAJcASEXe95SSu', name='calculate_age', type='function_call', id='fc_067fa566220c6710006a4f623f7e4c819aaea60f616585d768', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"amount":100}', call_id='call_UygfwsOjtIV4lSuZGqh8LOIc', name='convert_currency', type='function_call', id='fc_067fa566220c6710006a4f623f7e70819ab559d1d22c489a15', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"height":190,"weight":100}', call_id='call_gUs9Vnf4JYjyANSjD2uNhuOw', name='calculate_bmi', type='function_call', id='fc_067fa566220c6710006a4f623f7e78819a8197b668bc2d6ba6', namespace=None, status='completed')]

In [70]:
import json

if response.output:
    for tool_call in response.output:

        # 1. 추론 아이템(Reasoning) 처리
        if tool_call.type == 'reasoning':
            input_messages.append({
                'type': 'reasoning',
                'id' : tool_call.id,
                'content': tool_call.content if hasattr(tool_call, 'content') else [],
                'summary': tool_call.summary if hasattr(tool_call, 'summary') else []
            })
        
        # 2. 도구 호출(Function Call) 처리
        elif tool_call.type == 'function_call':
            # 중요 'id'와 'call_id'를 모두 명시하여 에러 방지
            input_messages.append({
                'type': 'function_call',
                'id' : tool_call.id,
                'call_id' : tool_call.id,  # 이 부분이 누락되면 에러 발생!
                'name' : tool_call.name,
                'arguments' : tool_call.arguments
            })
            
            print('호출도구 :', tool_call.name)

            args = json.loads(tool_call.arguments)
            result = None

            # 함수 이름에 맞게 실행
            if tool_call.name == 'calculate_age':
                result = calculate_age(args['birthdate'])
            elif tool_call.name == 'convert_currency':
                result = convert_currency(args['amount'])
            elif tool_call.name == 'calculate_bmi':
                result = calculate_bmi(args['height'], args['weight'])

            input_messages.append(
                {
                    'type' : 'function_call_output',
                    'call_id' : tool_call.call_id,
                    'output' : str(result)
                }
            )


호출도구 : calculate_age
호출도구 : convert_currency
호출도구 : calculate_bmi


In [71]:
input_messages

[{'role': 'user',
  'content': '내 생일은 1999-10-12일이야. 그리고 내가 100달러를 가지고 있는데 원화로 얼마인지 알려줘. 그리고 내 키가 190에 몸무게가 100kg인데 BMI 지수를 알려줘.'},
 {'type': 'function_call',
  'id': 'fc_067fa566220c6710006a4f623f7e4c819aaea60f616585d768',
  'call_id': 'fc_067fa566220c6710006a4f623f7e4c819aaea60f616585d768',
  'name': 'calculate_age',
  'arguments': '{"birthdate":"1999-10-12"}'},
 {'type': 'function_call_output',
  'call_id': 'call_vkm4kNFDdBGAJcASEXe95SSu',
  'output': '26'},
 {'type': 'function_call',
  'id': 'fc_067fa566220c6710006a4f623f7e70819ab559d1d22c489a15',
  'call_id': 'fc_067fa566220c6710006a4f623f7e70819ab559d1d22c489a15',
  'name': 'convert_currency',
  'arguments': '{"amount":100}'},
 {'type': 'function_call_output',
  'call_id': 'call_UygfwsOjtIV4lSuZGqh8LOIc',
  'output': '133000'},
 {'type': 'function_call',
  'id': 'fc_067fa566220c6710006a4f623f7e78819a8197b668bc2d6ba6',
  'call_id': 'fc_067fa566220c6710006a4f623f7e78819a8197b668bc2d6ba6',
  'name': 'calculate_bmi',
  'arguments': '

In [72]:
response_all = client.responses.create(
    model = 'gpt-5.5',
    input = input_messages,
    tools = fn_tools
)

print(response_all.output_text)

- 생일(1999-10-12) 기준 만 나이: **26세**
- 100달러 환산 금액: **133,000원**
- 키 190cm, 몸무게 100kg 기준 BMI: **27.7**

BMI 기준으로는 **과체중 범위**에 해당합니다.
